## Required Dataset Format

Labels should be in OBB format:
```
<class_id> <x1> <y1> <x2> <y2> <x3> <y3> <x4> <y4>
```

Example:
```
2 0.524 0.431 0.613 0.455 0.589 0.608 0.500 0.584
```

In [ ]:
import shutil
import os

folder_to_delete = '/content/training_data'

if os.path.exists(folder_to_delete):
    shutil.rmtree(folder_to_delete)
    print(f"Folder '{folder_to_delete}' and its contents deleted successfully.")
else:
    print(f"Folder '{folder_to_delete}' does not exist.")

In [ ]:
# Install the Ultralytics YOLO library
!pip install ultralytics

# Check if GPU is available
import torch
print(f"GPU Available: {torch.cuda.is_available()}")

In [ ]:
import zipfile
import os

zip_file_path = 'training_data.zip'
extract_dir = '/content/'

if not os.path.exists(zip_file_path):
    print(f"Error: {zip_file_path} not found.")
else:
    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        print(f"Dataset extracted successfully to {extract_dir}!")
    except zipfile.BadZipFile:
        print(f"Error: {zip_file_path} is not a valid zip file or is corrupted.")
    except Exception as e:
        print(f"An unexpected error occurred during extraction: {e}")

In [ ]:
# Verify dataset has OBB format (rotation angles)
import os
import sys

label_dir = '/content/training_data/labels'

# 1. Basic Path & File Validation
if not os.path.exists(label_dir):
    sys.exit("✗ Error: Labels directory not found.")

label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]
if not label_files:
    sys.exit("✗ Error: No .txt label files found.")

# 2. Content Extraction
with open(os.path.join(label_dir, label_files[0]), 'r') as f:
    first_line = f.readline().strip()

if not first_line:
    sys.exit("✗ Error: First label file is empty.")

# 3. Format Logic
parts = first_line.split()
num = len(parts)

print(f"Sample: {first_line} ({num} values)")

if num == 9:
    print("✓ OBB format confirmed (4 corner coordinates).")
else:
    print(f"✗ Unknown format detected.")

In [ ]:
import yaml

# Define the OBB config structure
data_config = {
    'path': '/content/training_data',  # Root dir
    'train': 'images',                 # Train images (relative to path)
    'val': 'images',                   # Val images (relative to path)

    # The dictionary of class names (IDs 0-10)
    'names': {
        0: 'X1-Y1-Z2',
        1: 'X1-Y2-Z1',
        2: 'X1-Y2-Z2',
        3: 'X1-Y2-Z2-CHAMFER',
        4: 'X1-Y2-Z2-TWINFILLET',
        5: 'X1-Y3-Z2',
        6: 'X1-Y3-Z2-FILLET',
        7: 'X1-Y4-Z1',
        8: 'X1-Y4-Z2',
        9: 'X2-Y2-Z2',
        10: 'X2-Y2-Z2-FILLET'
    }
}

# Write it to a file
with open('/content/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("Created /content/data.yaml for OBB training")

In [ ]:
from ultralytics import YOLO

# Load a pretrained OBB model (transfer learning)
model = YOLO('yolov8n-obb.pt')

# Train the model
results = model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    task='obb',  # specify OBB task
    project='/content/runs/train',
    name='papa_obb_model',
    degrees=45.0,  # Random rotation augmentation
    patience=50,   # Early stopping patience
)

print("✓ OBB Model training complete!")
print(f"Best model saved at: /content/runs/train/papa_obb_model/weights/best.pt")

Ultralytics 8.4.8 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=45.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=papa_obb_model2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=50, perspecti

In [ ]:
from IPython.display import Image, display

# Display a batch of predictions from the validation set
print("Checking OBB model predictions...")
display(Image(filename='/content/runs/train/papa_obb_model/val_batch0_pred.jpg', width=800))

# Display the Confusion Matrix
display(Image(filename='/content/runs/train/papa_obb_model/confusion_matrix.png', width=800))

# Display training curves
print("\nTraining Results:")
display(Image(filename='/content/runs/train/papa_obb_model/results.png', width=800))

In [ ]:
# Test the trained OBB model on a sample image
from ultralytics import YOLO
import cv2
from matplotlib import pyplot as plt
import os

model = YOLO('/content/runs/train/papa_obb_model/weights/best.pt')

# Get a test image
test_images = os.listdir('/content/training_data/images')
if test_images:
    test_img_path = os.path.join('/content/training_data/images', test_images[0])


    results = model.predict(test_img_path, conf=0.25)

    # Display results
    result_img = results[0].plot()  # Plot with OBB boxes

    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    plt.title('OBB Model Prediction (with rotation)')
    plt.axis('off')
    plt.show()

    # Print detection info
    if hasattr(results[0], 'obb') and results[0].obb is not None:
        print(f"✓ Detected {len(results[0].obb)} objects with OBB")
        print("Rotation angles are being predicted!")
    else:
        print("No OBB detections or standard boxes only")
else:
    print("No test images found!")

In [ ]:
from google.colab import files

# Download the trained OBB model
files.download('/content/runs/train/papa_obb_model/weights/best.pt')

print("✓ Downloaded best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>